In [50]:
%pip install "transformers>=4.41" datasets seqeval torch torchvision pillow pytesseract opencv-python shapely


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [51]:
import os
import json
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

from shapely.geometry import Polygon
import glob
from PIL import Image
from pytesseract import pytesseract
from lxml import etree
import ast

from sklearn.model_selection import train_test_split
pytesseract.tesseract_cmd = r'/opt/homebrew/bin/tesseract'

In [52]:
f = open('/Users/revathipriyan/Documents/Developer/REV CODES/document ai/project-3-at-2025-09-22-12-39-721c90c2.json')
label_studio_data = json.load(f)

In [53]:
def calculate_iou(box_1, box_2):
    poly_1 = Polygon(box_1)
    poly_2 = Polygon(box_2)
    # print(poly_1,poly_2)
    # iou = poly_1.intersection(poly_2).area / poly_1.union(poly_2).area
    iou = poly_1.intersection(poly_2).area
    min_area = min(poly_1.area,poly_2.area)
    return iou/min_area
    
    
def hocr_to_dataframe(fp):
    doc = etree.parse(fp)
    words = []
    wordConf = []
    coords_list = []
    for path in doc.xpath('//*'):
        if 'ocrx_word' in path.values():
            coord_text = path.values()[2].split(';')[0].split(' ')[1:] 
            word_coord = list(map(int, coord_text)) #x1, y1, x2, y2
            conf = [x for x in path.values() if 'x_wconf' in x][0]
            wordConf.append(int(conf.split('x_wconf ')[1]))
            words.append(path.text)
            coords_list.append(word_coord)

    dfReturn = pd.DataFrame({'word' : words,
                             'coords': coords_list,
                             'confidence' : wordConf})

    return(dfReturn)

In [54]:
document_data = dict()
document_data['file_name'] = []
document_data['labelled_bbox']= []

for i in range(len(label_studio_data)):
    row = label_studio_data[i]
    file_name = os.path.basename(row['data']['image'])
    label_list, labels, bboxes = [], [], []

    for label_ in row['annotations'][0]['result']:
        label_value = label_['value']
        x, y, w, h = label_value['x'], label_value['y'], label_value['width'], label_value['height']
        original_w , original_h = label_['original_width'], label_['original_height']

        x1 = int((x * original_w) / 100)
        y1 = int((y * original_h) / 100)
        x2 = x1 + int(original_w*w / 100)
        y2 = y1 + int(original_h*h / 100)
        
        label = label_value['rectanglelabels']
        label_list.append((label, (x1,y1,x2,y2), original_h, original_w))
        
    document_data['file_name'].append(file_name)    
    document_data['labelled_bbox'].append(label_list)        

custom_dataset = pd.DataFrame(document_data)

In [55]:
custom_dataset

,file_name,labelled_bbox
0,dcf8e7bb-batch1-1.png,"[([amount_no], (511, 766, 963, 811), 1236, 984..."
1,7437d4eb-batch1-2.png,"[([amount_no], (687, 908, 908, 941), 1200, 962..."
2,a4e9d297-batch1-3.png,"[([amount_no], (586, 583, 800, 608), 1248, 864..."
3,c2c0f3bd-batch1-4.png,"[([amount_no], (929, 1059, 1105, 1095), 1600, ..."
4,0854f2d9-batch1-0489.jpg,"[([amount_no], (1370, 1373, 1527, 1438), 2339,..."
5,daa6d313-batch1-0499.jpg,"[([amount_no], (1384, 1500, 1526, 1550), 2339,..."


In [1]:
label2id = {"invoice_no": 0, "date": 1, "amount": 2}
id2label = {v:k for k, v in label2id.items()}
id2label

{0: 'invoice_no', 1: 'date', 2: 'amount'}

In [57]:
import os, glob
# print label names expected
print("Sample label file_names (from dataset):")
print(custom_dataset['file_name'].head(20).tolist())

# list images found by common patterns
image_dirs = ['./data/input_images', './images', '/Users/revathipriyan/Documents/Developer/REV CODES/document ai/images']
patterns = [os.path.join(d, p) for d in image_dirs for p in ('*.jpg','*.jpeg','*.png','*.tif','*.tiff')]
found = []
for pat in patterns:
    found += glob.glob(pat)
found = sorted(set(found))
print(f"\nFound {len(found)} image files:")
for p in found[:50]:
    print(" ", os.path.basename(p), " <- ", p)

# which label entries don't match any found image by exact basename
image_basenames = {os.path.basename(p).lower() for p in found}
missing = []
for idx, row in custom_dataset.iterrows():
    file_name = row['file_name']
    if file_name.lower() not in image_basenames:
        missing.append(file_name)
print(f"\nLabel files without exact image match ({len(missing)}):")
for m in missing[:50]:
    print(" ", m)

# quick fuzzy check by stem (ignores extension)
image_stems = {os.path.splitext(os.path.basename(p))[0].lower(): p for p in found}
missing_by_stem = []
for idx, row in custom_dataset.iterrows():
    stem = os.path.splitext(row['file_name'])[0].lower()
    if stem not in image_stems:
        missing_by_stem.append(row['file_name'])
print(f"\nLabel files without stem match ({len(missing_by_stem)}):")
for m in missing_by_stem[:50]:
    print(" ", m)

Sample label file_names (from dataset):
['dcf8e7bb-batch1-1.png', '7437d4eb-batch1-2.png', 'a4e9d297-batch1-3.png', 'c2c0f3bd-batch1-4.png', '0854f2d9-batch1-0489.jpg', 'daa6d313-batch1-0499.jpg']

Found 12 image files:
  0854f2d9-batch1-0489.jpg  <-  ./images/0854f2d9-batch1-0489.jpg
  7437d4eb-batch1-2.png  <-  ./images/7437d4eb-batch1-2.png
  a4e9d297-batch1-3.png  <-  ./images/a4e9d297-batch1-3.png
  c2c0f3bd-batch1-4.png  <-  ./images/c2c0f3bd-batch1-4.png
  daa6d313-batch1-0499.jpg  <-  ./images/daa6d313-batch1-0499.jpg
  dcf8e7bb-batch1-1.png  <-  ./images/dcf8e7bb-batch1-1.png
  0854f2d9-batch1-0489.jpg  <-  /Users/revathipriyan/Documents/Developer/REV CODES/document ai/images/0854f2d9-batch1-0489.jpg
  7437d4eb-batch1-2.png  <-  /Users/revathipriyan/Documents/Developer/REV CODES/document ai/images/7437d4eb-batch1-2.png
  a4e9d297-batch1-3.png  <-  /Users/revathipriyan/Documents/Developer/REV CODES/document ai/images/a4e9d297-batch1-3.png
  c2c0f3bd-batch1-4.png  <-  /Users/rev

In [58]:
# Build images_map once: map basename (without extension) -> list of file paths
# This is resilient to .jpg/.png differences and avoids scanning the FS repeatedly.
import os, glob

image_dirs = ['./data/input_images', './images', '/Users/revathipriyan/Documents/Developer/REV CODES/document ai/images']
exts = ('*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff')

images_map = {}
for d in image_dirs:
    if not os.path.isdir(d):
        # skip missing directories
        continue
    for e in exts:
        pattern = os.path.join(d, e)
        for p in glob.glob(pattern):
            stem = os.path.splitext(os.path.basename(p))[0].lower()
            images_map.setdefault(stem, []).append(p)

found = sum(len(v) for v in images_map.values())
print(f"Image lookup prepared. Found {found} images across: {image_dirs}")

# Usage example (inside your main loop):
# file_name = i[1]['file_name']
# file_stem = os.path.splitext(file_name)[0].lower()
# if file_stem in images_map:
#     for image in images_map[file_stem]:
#         # process image
# else:
#     print(f"No image found for {file_name}")


Image lookup prepared. Found 12 images across: ['./data/input_images', './images', '/Users/revathipriyan/Documents/Developer/REV CODES/document ai/images']


In [59]:
# Safe helper to produce hOCR from an image using tesseract CLI
# Returns path to .hocr on success, or None on failure.
import os, subprocess, traceback
from pytesseract import pytesseract

def produce_hocr(image_path, base_name):
    """Run tesseract to produce base_name.hocr. Returns path or None on failure."""
    outdir = os.path.dirname(base_name)
    os.makedirs(outdir, exist_ok=True)

    # Use the tesseract CLI explicitly: tesseract input_file output_base hocr
    cmd = [pytesseract.tesseract_cmd, image_path, base_name, 'hocr']
    try:
        print('Running tesseract ->', ' '.join(cmd))
        cp = subprocess.run(cmd, capture_output=True, text=True)
        if cp.returncode != 0:
            print('tesseract returned non-zero exit code')
            print('stdout:', cp.stdout)
            print('stderr:', cp.stderr)
            return None
    except FileNotFoundError:
        print('tesseract binary not found at', pytesseract.tesseract_cmd)
        return None
    except Exception:
        print('Exception running tesseract:')
        traceback.print_exc()
        return None

    hocr_file = base_name + '.hocr'
    if not os.path.exists(hocr_file):
        print('Expected hOCR file not found after tesseract:', hocr_file)
        return None
    return hocr_file

# Helper to safely parse hocr into dataframe
def safe_hocr_to_dataframe(hocr_path):
    try:
        return hocr_to_dataframe(hocr_path)
    except Exception:
        print('Failed to parse hOCR:', hocr_path)
        traceback.print_exc()
        return None


In [60]:
%%time

final_list = []
    
for i in tqdm(custom_dataset.iterrows(), total=custom_dataset.shape[0]):
    custom_label_text = {}
    word_list = []
    ner_tags_list  = []
    bboxes_list = []
    
    file_name = i[1]['file_name']
    file_stem = os.path.splitext(file_name)[0].lower()

    if file_stem in images_map:
        for image in images_map[file_stem]:
            frame_file_name = os.path.basename(image)
            custom_label_text['id'] = i[0]
            image_basename = os.path.basename(image)
            custom_label_text['file_name'] = image_basename
            annotations = []
            label_coord_list = i[1]['labelled_bbox']
            for label_coord in label_coord_list:
                (x1,y1,x2,y2) = label_coord[1]
                box1 = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]] 
                label = label_coord[0][0]
                base_name = os.path.join('./data', 'layoutlmv3_hocr_output',os.path.basename(image).split('.')[0])

                # produce hocr safely and parse it
                hocr_path = produce_hocr(image, base_name)
                if hocr_path is None:
                    print(f"Skipping OCR for {image} (hocr not produced)")
                    continue
                hocr_df = safe_hocr_to_dataframe(hocr_path)
                if hocr_df is None:
                    print(f"Skipping parsing for {hocr_path}")
                    continue

                for word in hocr_df.iterrows():
                    coords = word[1]['coords']
                    (x1df,y1df,x2df,y2df) = coords
                    box2 = [[x1df, y1df], [x2df, y1df], [x2df, y2df], [x1df, y2df]]
                    words = word[1]['word']
                    overlap_perc = calculate_iou(box1,box2)
                    temp_dic = {}
                    if overlap_perc > 0.80:
                        if words != '-':
                            word_list.append(words)
                            bboxes_list.append(coords)
                            # guard label lookup
                            if label not in label2id:
                                print(f"Unknown label '{label}' — skipping")
                                continue
                            label_id = label2id[label]
                            ner_tags_list.append(label_id)
                        
                        custom_label_text['tokens'] = word_list
                        custom_label_text['bboxes'] = bboxes_list
                        custom_label_text['ner_tags'] = ner_tags_list

    else:
        print(f"Warning: no image found for {file_name}")

    final_list.append(custom_label_text)
    print("final list",final_list)


  0%|          | 0/6 [00:00<?, ?it/s]

Running tesseract -> /opt/homebrew/bin/tesseract ./images/dcf8e7bb-batch1-1.png ./data/layoutlmv3_hocr_output/dcf8e7bb-batch1-1 hocr
Unknown label 'amount_no' — skipping
Unknown label 'amount_no' — skipping
Running tesseract -> /opt/homebrew/bin/tesseract ./images/dcf8e7bb-batch1-1.png ./data/layoutlmv3_hocr_output/dcf8e7bb-batch1-1 hocr
Unknown label 'amount_no' — skipping
Unknown label 'amount_no' — skipping
Running tesseract -> /opt/homebrew/bin/tesseract ./images/dcf8e7bb-batch1-1.png ./data/layoutlmv3_hocr_output/dcf8e7bb-batch1-1 hocr
Running tesseract -> /opt/homebrew/bin/tesseract ./images/dcf8e7bb-batch1-1.png ./data/layoutlmv3_hocr_output/dcf8e7bb-batch1-1 hocr
Running tesseract -> /opt/homebrew/bin/tesseract ./images/dcf8e7bb-batch1-1.png ./data/layoutlmv3_hocr_output/dcf8e7bb-batch1-1 hocr
Running tesseract -> /opt/homebrew/bin/tesseract /Users/revathipriyan/Documents/Developer/REV CODES/document ai/images/dcf8e7bb-batch1-1.png ./data/layoutlmv3_hocr_output/dcf8e7bb-batch1-

 17%|█▋        | 1/6 [00:02<00:11,  2.28s/it]

final list [{'id': 0, 'file_name': 'dcf8e7bb-batch1-1.png', 'tokens': ['ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864', 'ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864'], 'bboxes': [[564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55], [564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55]], 'ner_tags': [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0]}]
Running tesseract -> /opt/homebrew/bin/tesseract ./images/7437d4eb-batch1-2.png ./data/layoutlmv3_hocr_output/7437d4eb-batch1-2 hocr
Unknown label 'amount_no' — skipping
Unknown label 'amount_no' — skipping
Unknown label 'amount_no' — skipping
Running tesseract -> /opt/homebrew/bin/tesseract ./images/7437d4eb-batch1-2.png ./data/layoutlmv3_hocr_output/7437d4eb-batch1-2

 33%|███▎      | 2/6 [00:04<00:09,  2.34s/it]

...

final list [{'id': 0, 'file_name': 'dcf8e7bb-batch1-1.png', 'tokens': ['ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864', 'ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864'], 'bboxes': [[564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55], [564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55]], 'ner_tags': [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0]}, {'id': 1, 'file_name': '7437d4eb-batch1-2.png', 'tokens': ['TOTAL', '$', '86.43', 'DATE:', '14', 'MAY,', '2015', 'INVOICE', '#554951', 'TOTAL', '$', '86.43', 'DATE:', '14', 'MAY,', '2015', 'INVOICE', '#554951'], 'bboxes': [[701, 922, 770, 937], [819, 920, 831, 939], [839, 922, 895, 937], [697, 178, 758, 193], [764, 178, 787, 192], [794, 178, 844, 196], [850, 178, 896, 

 83%|████████▎ | 5/6 [00:10<00:02,  2.17s/it]

final list [{'id': 0, 'file_name': 'dcf8e7bb-batch1-1.png', 'tokens': ['ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864', 'ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864'], 'bboxes': [[564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55], [564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55]], 'ner_tags': [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0]}, {'id': 1, 'file_name': '7437d4eb-batch1-2.png', 'tokens': ['TOTAL', '$', '86.43', 'DATE:', '14', 'MAY,', '2015', 'INVOICE', '#554951', 'TOTAL', '$', '86.43', 'DATE:', '14', 'MAY,', '2015', 'INVOICE', '#554951'], 'bboxes': [[701, 922, 770, 937], [819, 920, 831, 939], [839, 922, 895, 937], [697, 178, 758, 193], [764, 178, 787, 192], [794, 178, 844, 196], [850, 178, 896, 

100%|██████████| 6/6 [00:13<00:00,  2.31s/it]

final list [{'id': 0, 'file_name': 'dcf8e7bb-batch1-1.png', 'tokens': ['ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864', 'ND', 'TOTA', 'Invoice', 'Date', '07-FEB-2022', 'Invoice', 'Number', '2101825864'], 'bboxes': [[564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55], [564, 785, 577, 795], [581, 785, 613, 795], [642, 63, 679, 71], [684, 62, 708, 71], [824, 62, 891, 71], [642, 46, 679, 55], [684, 46, 726, 55], [824, 46, 890, 55]], 'ner_tags': [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0]}, {'id': 1, 'file_name': '7437d4eb-batch1-2.png', 'tokens': ['TOTAL', '$', '86.43', 'DATE:', '14', 'MAY,', '2015', 'INVOICE', '#554951', 'TOTAL', '$', '86.43', 'DATE:', '14', 'MAY,', '2015', 'INVOICE', '#554951'], 'bboxes': [[701, 922, 770, 937], [819, 920, 831, 939], [839, 922, 895, 937], [697, 178, 758, 193], [764, 178, 787, 192], [794, 178, 844, 196], [850, 178, 896, 

In [61]:
train, test = train_test_split(final_list, random_state=21, test_size=0.3)

for detail  in final_list:
    with open('final_list_text.txt', 'a') as f:
        f.write(str(detail))
        f.write('\n')
        
for detail  in train:
    with open('train.txt', 'a') as f:
        f.write(str(detail))
        f.write('\n')
        
for detail  in test:
    with open('test.txt', 'a') as f:
        f.write(str(detail))
        f.write('\n')

In [62]:
final_list

[{'id': 0,
  'file_name': 'dcf8e7bb-batch1-1.png',
  'tokens': ['ND',
   'TOTA',
   'Invoice',
   'Date',
   '07-FEB-2022',
   'Invoice',
   'Number',
   '2101825864',
   'ND',
   'TOTA',
   'Invoice',
   'Date',
   '07-FEB-2022',
   'Invoice',
   'Number',
   '2101825864'],
  'bboxes': [[564, 785, 577, 795],
   [581, 785, 613, 795],
   [642, 63, 679, 71],
   [684, 62, 708, 71],
   [824, 62, 891, 71],
   [642, 46, 679, 55],
   [684, 46, 726, 55],
   [824, 46, 890, 55],
   [564, 785, 577, 795],
   [581, 785, 613, 795],
   [642, 63, 679, 71],
   [684, 62, 708, 71],
   [824, 62, 891, 71],
   [642, 46, 679, 55],
   [684, 46, 726, 55],
   [824, 46, 890, 55]],
  'ner_tags': [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0]},
 {'id': 1,
  'file_name': '7437d4eb-batch1-2.png',
  'tokens': ['TOTAL',
   '$',
   '86.43',
   'DATE:',
   '14',
   'MAY,',
   '2015',
   'INVOICE',
   '#554951',
   'TOTAL',
   '$',
   '86.43',
   'DATE:',
   '14',
   'MAY,',
   '2015',
   'INVOICE',
   '#554951'],
  'bboxes': [[70